<a href="https://colab.research.google.com/github/DanisStepanov/image_processing/blob/main/%D0%9F%D1%80%D0%BE%D0%B5%D0%BA%D1%82%D0%BD%D0%B0%D1%8F_%D1%80%D0%B0%D0%B1%D0%BE%D1%82%D0%B0_%E2%84%965_%D0%90%D0%BD%D0%B0%D0%BB%D0%B8%D0%B7_%D0%B8_%D1%81%D1%80%D0%B0%D0%B2%D0%BD%D0%B5%D0%BD%D0%B8%D0%B5_%D0%B4%D0%B5%D1%82%D0%B5%D0%BA%D1%82%D0%BE%D1%80%D0%BE%D0%B2_%D0%BA%D0%BB%D1%8E%D1%87%D0%B5%D0%B2%D1%8B%D1%85_%D1%82%D0%BE%D1%87%D0%B5%D0%BA_%D0%B8_%D0%B4%D0%B5%D1%81%D0%BA%D1%80%D0%B8%D0%BF%D1%82%D0%BE%D1%80%D0%BE%D0%B2_%D0%B2_OpenCV.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Проектная работа №5. Анализ и сравнение детекторов ключевых точек и дескрипторов в OpenCV**

## **1. Цель работы**

Экспериментально сравнить детекторы и дескрипторы ключевых точек (SIFT, ORB, AKAZE и комбинацию FAST + SIFT) на наборах снимков с **известной геометрией**, где правильность каждой найденной пары можно проверить, и сформулировать обоснованные числами рекомендации: какой метод выбирать для искажённых, мультимодальных и специальных изображений.

## **2. Задачи работы**

1. **Сравнить** методы на синтетических искажениях с регулируемой силой (размытие, сжатие, освещение, поворот и масштаб) и проверить, предсказывают ли синтетические опыты поведение методов на реальных искажённых снимках (Задание 1).
2. **Исследовать** сопоставление снимков разных спектральных диапазонов (видимый и ближний инфракрасный) и влияние предобработки (Задание 2).
3. **Оценить** методы в одном из специальных доменов на выбор: регистрация медицинских снимков сетчатки, повторяющиеся структуры или привязка кадров с дрона к спутниковой подложке (Задание 3).
4. **Свести** результаты в таблицы и графики, **сделать выводы** и **дать рекомендации** по выбору методов.

---

## **3. Данные**

Все данные — открытые наборы снимков, для которых известно точное соответствие между изображениями. Поэтому для каждой найденной пары ключевых точек можно проверить, верна она или нет, а не полагаться на внешнее сходство. Все наборы загружаются кодом прямо в блокноте; искать изображения самостоятельно не нужно.

| Набор | Что содержит | Эталон | Где используется |
|---|---|---|---|
| [**Oxford Affine**](https://www.robots.ox.ac.uk/~vgg/research/affine/) (Visual Geometry Group, Оксфорд; Mikolajczyk et al., 2005) | 8 серий по 6 снимков одной плоской сцены с нарастающим искажением: `graf`, `wall` — смена ракурса; `boat`, `bark` — масштаб и поворот; `bikes`, `trees` — размытие; `leuven` — освещение; `ubc` — сжатие JPEG. Около 50 МБ | Гомографии из снимка 1 в снимки 2–6 (файлы `H1to2p` … `H1to6p`) | Задание 1, вариант 3.2 |
| [**EPFL RGB–NIR Scene Dataset**](https://www.epfl.ch/labs/ivrl/research/downloads/rgb-nir-scene-dataset/) (Brown, Süsstrunk, CVPR 2011) | 477 пар «видимый диапазон (RGB) / ближний инфракрасный (NIR)» в 9 категориях: `country`, `field`, `forest`, `indoor`, `mountain`, `oldbuilding`, `street`, `urban`, `water`. Снимки около 1024×680 пикселей (часть — вертикальные) | Пары сняты совмещёнными: геометрия RGB и NIR одинакова, поэтому к снимку NIR применяется известная гомография | Задание 2 |
| [**FIRE — Fundus Image Registration Dataset**](https://projects.ics.forth.gr/cvrl/fire/) (Hernandez-Matas et al., 2017) | 134 пары цветных снимков глазного дна 2912×2912 пикселей. Группы: **S** — 71 пара с большим перекрытием, **P** — 49 пар с малым перекрытием, **A** — 14 пар с анатомическими изменениями (снятых с интервалом во времени). Архив 7z, 277 МБ | Для каждой пары — 10 соответствующих точек, размеченных вручную | Вариант 3.1 |
| [**OpenDroneMap, набор Toledo**](https://github.com/OpenDroneMap/odm_data_toledo) | 87 кадров съёмки с дрона DJI (4000×3000 пикселей) над парком, зданиями и полями; в метаданных EXIF — координаты и высота съёмки | Координаты камеры по GPS и ширина кадра по высоте съёмки | Задание 1 (скорость), вариант 3.3 |
| [**Esri World Imagery**](https://www.arcgis.com/home/item.html?id=10df2279f9684e4a9f6a7f08febac2a9) и Esri World Street Map | Тайлы спутниково-аэрофотосъёмочной подложки и векторной карты одного и того же района | Тайлы снимка и карты совмещены по построению | Вариант 3.3, дополнительное задание 2 |

**Почему не искать изображения в интернете.** Для двух случайных снимков неизвестно, где на втором находится каждая точка первого. Тогда «правильными» приходится считать пары, которые прошёл RANSAC, а RANSAC сам может ошибиться — и все метрики окажутся оценкой RANSAC, а не дескриптора. В наборах выше эталон задан независимо от сопоставления.

> 📚 **Подсказка:** работа опирается на [**Теория №7: вспомогательные функции**](https://colab.research.google.com/drive/1HZSncySLe9Me1pLhDUWMU_W15F8vUZfD#scrollTo=f3828da1), [**Теория №7: 2.7. Сопоставление**](https://colab.research.google.com/drive/1HZSncySLe9Me1pLhDUWMU_W15F8vUZfD#scrollTo=fe29c5b2), [**Теория №7: 2.8. Гомография и RANSAC**](https://colab.research.google.com/drive/1HZSncySLe9Me1pLhDUWMU_W15F8vUZfD#scrollTo=e584fd36) и [**Теория №7: 2.9. Сравнительный анализ**](https://colab.research.google.com/drive/1HZSncySLe9Me1pLhDUWMU_W15F8vUZfD#scrollTo=49251e77). Код из теории можно переиспользовать.

> 📝 **Как выполнять работу.** Выполняйте ячейки по порядку. Ячейки с готовым кодом запускайте без изменений: они загружают данные и содержат общие функции. В ячейках `# Ваш код` напишите код по заданию; имена функций и переменных, указанные в задании, используются в следующих пунктах. Ответы пишите в текстовых ячейках `**Ответ:**` (двойной щелчок по ячейке → текст → Shift+Enter) с опорой на полученные числа. Выполнение всех ячеек занимает от 5 до 10 минут в зависимости от варианта Задания 3.

---

## **4. Подготовка рабочей среды**

### **4.1. Установка библиотек**

В Colab предустановлен OpenCV 5, где AKAZE вынесен из основной библиотеки. Установим OpenCV 4.14, а также `remotezip` (чтение отдельных файлов из большого архива по сети) и `py7zr` (распаковка архивов 7z). Ячейку нужно выполнить первой; если OpenCV уже был импортирован в этом сеансе, после установки перезапустите сеанс («Среда выполнения» → «Перезапустить сеанс»). Подробнее: [**Теория №7: установка OpenCV**](https://colab.research.google.com/drive/1HZSncySLe9Me1pLhDUWMU_W15F8vUZfD#scrollTo=17b4e1e9).

In [ ]:
%pip install -q opencv-python==4.14.0.94 opencv-python-headless==4.14.0.94 opencv-contrib-python==4.14.0.94 remotezip py7zr

In [ ]:
import os
import re
import glob
import math
import time
import shutil
import tarfile
import urllib.request

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import py7zr
from PIL import Image
from remotezip import RemoteZip

%config InlineBackend.figure_format = 'jpeg'
print('OpenCV', cv2.__version__)

### **4.2. Загрузка серий Oxford Affine**

Функция `download` скачивает файл в папку `data`, если его там ещё нет. Серии Oxford Affine нужны в Задании 1, поэтому загрузим их сразу; остальные наборы загружаются в своих заданиях.

In [ ]:
os.makedirs('data', exist_ok=True)

def download(url, name):
    """Скачивает файл в папку data, если его там ещё нет, и возвращает путь к нему."""
    path = os.path.join('data', name)
    if not os.path.exists(path):
        request = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(request) as response, open(path + '.part', 'wb') as file:
            shutil.copyfileobj(response, file)
        os.replace(path + '.part', path)  # недокачанный файл не будет принят за готовый
    return path

OXFORD_SERIES = {
    'graf': 'смена ракурса', 'wall': 'смена ракурса (кирпичная стена)',
    'boat': 'масштаб и поворот', 'bark': 'масштаб и поворот (кора)',
    'bikes': 'размытие', 'trees': 'размытие (листва)',
    'leuven': 'освещение', 'ubc': 'сжатие JPEG',
}
for seq in OXFORD_SERIES:
    archive = download(f'https://thor.robots.ox.ac.uk/datasets/affine/{seq}.tar.gz', f'{seq}.tar.gz')
    if not os.path.isdir(f'data/{seq}'):
        tarfile.open(archive).extractall(f'data/{seq}', filter='data')
print(sorted(os.listdir('data')))

### **4.3. Общие функции из теории**

Эти функции взяты из теории ([**Теория №7: вспомогательные функции**](https://colab.research.google.com/drive/1HZSncySLe9Me1pLhDUWMU_W15F8vUZfD#scrollTo=f3828da1)) и используются во всех заданиях: чтение снимков и эталонных гомографий серий Oxford, перевод в оттенки серого, перенос точек гомографией, ошибка найденной гомографии и рисование пар (верные — зелёным, ошибочные — красным).

In [ ]:
def oxford(seq, i):
    """Снимок i серии Oxford Affine в формате BGR."""
    return cv2.imread(glob.glob(f'data/{seq}/img{i}.p?m')[0])

def oxford_H(seq, k):
    """Эталонная гомография из снимка 1 в снимок k."""
    return np.loadtxt(f'data/{seq}/H1to{k}p')

def gray(img):
    return img if img.ndim == 2 else cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

def rgb(img):
    return cv2.cvtColor(img, cv2.COLOR_GRAY2RGB) if img.ndim == 2 else cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def transfer_error(pts1, pts2, H):
    """Расстояние (в пикселях) между точкой, перенесённой гомографией H, и её найденной парой."""
    projected = cv2.perspectiveTransform(pts1.reshape(-1, 1, 2).astype(np.float32), H).reshape(-1, 2)
    return np.linalg.norm(projected - pts2, axis=1)

def corner_error(H_found, H_true, shape):
    """Среднее расхождение (в пикселях) углов кадра, перенесённых найденной и эталонной гомографией."""
    h, w = shape[:2]
    corners = np.float32([[0, 0], [w, 0], [w, h], [0, h]]).reshape(-1, 1, 2)
    diff = cv2.perspectiveTransform(corners, H_found) - cv2.perspectiveTransform(corners, H_true)
    return float(np.linalg.norm(diff, axis=2).mean())

def draw_checked(img1, kp1, img2, kp2, good, correct, n=60):
    """Рисует до n пар: верные — зелёным, ошибочные — красным."""
    idx = np.linspace(0, len(good) - 1, min(n, len(good))).astype(int)
    wrong = [good[i] for i in idx if not correct[i]]
    right = [good[i] for i in idx if correct[i]]
    out = cv2.drawMatches(img1, kp1, img2, kp2, wrong, None, matchesThickness=2, matchColor=(0, 0, 255),
                          flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
    return cv2.drawMatches(img1, kp1, img2, kp2, right, out, matchesThickness=2, matchColor=(0, 255, 0),
                           flags=cv2.DrawMatchesFlags_DRAW_OVER_OUTIMG | cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)

### **4.4. Измерение времени**

**1. Напишите функцию `measure_time(func, *args, repeats=3)`**, которая вызывает `func(*args)` `repeats` раз и возвращает результат последнего вызова и **медиану** времени в секундах. Время измеряйте функцией `time.perf_counter()`: она точнее `time.time()`. Медиана устойчивее среднего к единичным задержкам (например, первому «холодному» запуску).

In [ ]:
# Ваш код
# def measure_time(func, *args, repeats=3):
#     ...  # вызвать func(*args) repeats раз, замерить время time.perf_counter()
#     return result, медиана_времени

---

## **5. Задание 1. Сравнительный анализ методов в экстремальных условиях**

В этом задании вы сами создаёте искажения с регулируемой силой и знаете точное преобразование между исходным и искажённым снимком. Это позволяет измерить, как быстро метод теряет верные пары при усилении искажения. Затем вы проверите главное: **переносятся ли выводы, полученные на синтетических искажениях, на реальные снимки** с теми же видами искажений из серий Oxford Affine.

### **5.1. Гипотезы**

Прежде чем запускать эксперименты, сформулируйте 2 гипотезы о том, какой метод выиграет или проиграет при определённом искажении, и объясните почему, опираясь на устройство методов: [**Теория №7: 2.4. ORB**](https://colab.research.google.com/drive/1HZSncySLe9Me1pLhDUWMU_W15F8vUZfD#scrollTo=f884999e), [**Теория №7: 2.5. SIFT**](https://colab.research.google.com/drive/1HZSncySLe9Me1pLhDUWMU_W15F8vUZfD#scrollTo=ce7d87bd), [**Теория №7: 2.6. AKAZE**](https://colab.research.google.com/drive/1HZSncySLe9Me1pLhDUWMU_W15F8vUZfD#scrollTo=9b533304), [**Теория №7: 2.3. FAST**](https://colab.research.google.com/drive/1HZSncySLe9Me1pLhDUWMU_W15F8vUZfD#scrollTo=bd047846).

| № | Свойство метода (из теории) | Гипотеза | Как проверить |
|---|---|---|---|
| 1 | ORB сравнивает яркости отдельных пар пикселей окрестности | Блочные артефакты сильного сжатия JPEG собьют сравнения, и ORB потеряет верные пары быстрее SIFT | Доля верных пар при качестве JPEG 5 |
| 2 | ... | ... | ... |
| 3 | ... | ... | ... |

**Ответ (ваши гипотезы 2–3):** _______________

### **5.2. Функции искажений**

**1. Напишите функции четырёх искажений.** Для каждого из них в сериях Oxford Affine есть реальный аналог, с которым синтетику можно будет сравнить в пункте 5.6. Каждая функция принимает полутоновое изображение `uint8` и параметр силы искажения и возвращает изображение того же размера и типа:

- `gaussian_blur(img, sigma)` — гауссово размытие с параметром σ (`cv2.GaussianBlur` с размером ядра `(0, 0)`);
- `darken(img, alpha)` — затемнение: умножение яркости на `alpha` (`cv2.convertScaleAbs`);
- `jpeg(img, quality)` — сжатие JPEG с качеством `quality` и обратное декодирование (`cv2.imencode` и `cv2.imdecode`);
- `rotate_scale(img, params)` — поворот на угол `params[0]` (в градусах) и масштаб `params[1]` вокруг центра (`cv2.getRotationMatrix2D`, `cv2.warpAffine`).

**2. Напишите функцию `distort(name, img, level)`**, которая применяет искажение `name` из словаря `DISTORTIONS` ниже и возвращает пару `(искажённое изображение, H)`, где `H` — эталонная гомография 3×3 из исходного снимка в искажённый. Для фотометрических искажений (размытие, затемнение, JPEG) геометрия не меняется, поэтому `H` — единичная матрица. Для поворота и масштаба `H` — матрица аффинного преобразования 2×3, дополненная строкой `[0, 0, 1]`.

In [ ]:
# Ваш код
# def gaussian_blur(img, sigma): ...
# def darken(img, alpha): ...
# def jpeg(img, quality): ...
# def rotate_scale(img, params): ...   # возвращает (изображение, H 3×3)

# Искажение: функция и три уровня силы — от слабого к сильному (уровни можно не менять)
# DISTORTIONS = {
#     'гауссово размытие': (gaussian_blur, [2, 4, 6]),
#     'затемнение': (darken, [0.6, 0.3, 0.15]),
#     'JPEG': (jpeg, [30, 10, 5]),
#     'поворот и масштаб': (rotate_scale, [(15, 0.9), (45, 0.7), (90, 0.5)]),
# }

# def distort(name, img, level):
#     ...  # вернуть (искажённое изображение, H)

**3. Выберите три базовых снимка** — первые снимки серий Oxford Affine из списка `graf`, `wall`, `boat`, `bark`, `bikes`, `trees`, `leuven`, `ubc` — и запишите их имена в список `BASES`. Покажите для одного из них все четыре искажения на среднем уровне силы.

In [ ]:
# Ваш код
# BASES = [...]   # три серии из списка
# Покажите исходный снимок BASES[0] и четыре искажения на среднем уровне (уровень 2)

### **5.3. Пайплайн «детектор + дескриптор + сопоставление»**

Сравниваются четыре комбинации: SIFT, ORB и AKAZE (у каждого свой детектор и дескриптор) и **FAST + SIFT** — быстрый детектор углов без масштаба и ориентации с дескриптором SIFT. Последняя комбинация показывает, что даёт дескриптору хороший детектор. Подробнее о методах: [**Теория №7: 2.4. ORB**](https://colab.research.google.com/drive/1HZSncySLe9Me1pLhDUWMU_W15F8vUZfD#scrollTo=f884999e), [**Теория №7: 2.5. SIFT**](https://colab.research.google.com/drive/1HZSncySLe9Me1pLhDUWMU_W15F8vUZfD#scrollTo=ce7d87bd), [**Теория №7: 2.6. AKAZE**](https://colab.research.google.com/drive/1HZSncySLe9Me1pLhDUWMU_W15F8vUZfD#scrollTo=9b533304), [**Теория №7: 2.3. FAST**](https://colab.research.google.com/drive/1HZSncySLe9Me1pLhDUWMU_W15F8vUZfD#scrollTo=bd047846).

**1. Напишите функцию `create_method(name)`**, которая возвращает тройку `(detector, descriptor, norm)` для `name` из списка `METHOD_NAMES = ['SIFT', 'ORB', 'AKAZE', 'FAST+SIFT']`:
- для SIFT, ORB и AKAZE детектором и дескриптором служит один и тот же объект; ORB создайте с `nfeatures=5000` (500 по умолчанию слишком мало для снимков такого размера);
- для FAST + SIFT детектор — `cv2.FastFeatureDetector_create(threshold=20)`, дескриптор — `cv2.SIFT_create()`;
- `norm` — `cv2.NORM_L2` для вещественных дескрипторов (SIFT) и `cv2.NORM_HAMMING` для бинарных.

Создайте словарь `METHODS = {name: create_method(name) for name in METHOD_NAMES}`.

**2. Напишите функцию `detect_and_describe(img, name, max_points=5000, mask=None)`**: найти точки детектором (`detector.detect(img, mask)`), оставить не более `max_points` точек с наибольшим откликом (`kp.response`) и вычислить для них дескрипторы (`descriptor.compute(img, kps)`). Функция возвращает пару `(kps, des)`. Ограничение числа точек уравнивает методы: без него FAST возвращает десятки тысяч точек. Параметр `mask` понадобится в варианте 3.1.

**3. Напишите функцию `match_descriptors(des1, des2, norm, ratio=0.8)`**: перебор (`cv2.BFMatcher(norm).knnMatch(des1, des2, k=2)`) и тест Лоу с порогом `ratio`. Возвращает список пар `cv2.DMatch`, прошедших тест. Учтите, что для некоторых точек `knnMatch` может вернуть меньше двух соседей. Подробнее: [**Теория №7: тест Лоу**](https://colab.research.google.com/drive/1HZSncySLe9Me1pLhDUWMU_W15F8vUZfD#scrollTo=7795d677).

In [ ]:
# Ваш код
# METHOD_NAMES = ['SIFT', 'ORB', 'AKAZE', 'FAST+SIFT']
# def create_method(name): ...                      # -> (detector, descriptor, norm)
# METHODS = {name: create_method(name) for name in METHOD_NAMES}
# def detect_and_describe(img, name, max_points=5000, mask=None): ...   # -> (kps, des)
# def match_descriptors(des1, des2, norm, ratio=0.8): ...              # -> список cv2.DMatch

### **5.4. Метрики**

Все метрики считаются по **эталонной** гомографии, а не по результату RANSAC: инлайеры RANSAC — это пары, согласованные с найденной моделью, и при ошибке RANSAC они не совпадают с верными парами. Пример такого расхождения разобран в теории: [**Теория №7: проверка пар по эталонной гомографии**](https://colab.research.google.com/drive/1HZSncySLe9Me1pLhDUWMU_W15F8vUZfD#scrollTo=69332f28).

**Повторяемость.** Доля точек снимка 1, которые после переноса эталонной гомографией попадают на снимок 2 и имеют там найденную точку не дальше 3 пикселей. Чтобы плотные детекторы не получали преимущества просто за счёт числа точек, у обоих снимков берутся одинаковые 1000 точек с наибольшим откликом. Повторяемость оценивает только детектор; функция `repeatability` ниже дана готовой.

**Верные пары и их доля.** Пара после теста Лоу верна, если точка снимка 1, перенесённая эталонной гомографией, лежит не дальше 3 пикселей от своей пары на снимке 2 (функция `transfer_error`). Доля верных — их доля среди всех пар после теста Лоу.

**Ошибка найденной гомографии.** По парам после теста Лоу оценивается гомография (`cv2.findHomography`, RANSAC, порог 3 пикселя) и сравнивается с эталонной: функция `corner_error` возвращает среднее расхождение углов кадра. Пару снимков считаем **совмещённой**, если ошибка меньше 5 пикселей.

In [ ]:
def repeatability(kp1, kp2, H, shape2, top=1000, radius=3):
    """Доля из top лучших точек снимка 1, повторившихся на снимке 2 не дальше radius пикселей."""
    if len(kp1) == 0 or len(kp2) == 0:
        return 0.0
    pts1 = np.float32([k.pt for k in sorted(kp1, key=lambda k: -k.response)[:top]])
    pts2 = np.float32([k.pt for k in sorted(kp2, key=lambda k: -k.response)[:top]])
    proj = cv2.perspectiveTransform(pts1.reshape(-1, 1, 2), H).reshape(-1, 2)
    h, w = shape2[:2]
    inside = (proj[:, 0] >= 0) & (proj[:, 0] < w) & (proj[:, 1] >= 0) & (proj[:, 1] < h)
    dist = np.linalg.norm(proj[inside, None] - pts2[None], axis=2).min(axis=1)  # до ближайшей точки снимка 2
    return float((dist <= radius).mean())

**1. Напишите функцию `evaluate_pair(feat1, feat2, H_true, shape, norm, ratio=0.8)`**, где `feat1` и `feat2` — результаты `detect_and_describe`. Функция возвращает словарь `{'пар': ..., 'верных': ..., 'доля верных': ..., 'ошибка H': ...}`. Если пар меньше четырёх, гомографию оценить нельзя: верните ошибку `np.inf`. Учтите, что на сильно искажённом снимке детектор может не найти ни одной точки: тогда `descriptor.compute` возвращает `des = None`, и сопоставлять нечего.

In [ ]:
# Ваш код
# def evaluate_pair(feat1, feat2, H_true, shape, norm, ratio=0.8): ...   # -> словарь метрик

# Проверьте функции на паре graf 1 → 3 методом SIFT

### **5.5. Эксперимент на синтетических искажениях**

**1. Для каждого базового снимка из `BASES`, каждого искажения, каждого из трёх уровней и каждого метода** вычислите метрики `evaluate_pair` и повторяемость (`repeatability`) и соберите их в таблицу `synthetic` (`pandas.DataFrame`) со столбцами `снимок`, `искажение`, `уровень` (1, 2, 3), `метод` и метриками.

Признаки исходного снимка не зависят от искажения: вычислите их один раз для каждого снимка и метода и используйте для всех искажений — это вдвое сокращает время эксперимента (около 40 секунд).

In [ ]:
# Ваш код
# rows = []
# for base in BASES:
#     ...  # признаки исходного снимка для каждого метода — один раз
#     for dist_name, (_, levels) in DISTORTIONS.items():
#         for level_no, level in enumerate(levels, start=1):
#             ...  # искажение, признаки искажённого снимка, evaluate_pair, repeatability
# synthetic = pd.DataFrame(rows)

**2. Постройте графики**: для каждого искажения — долю верных пар (среднюю по трём снимкам) в зависимости от уровня, отдельной линией для каждого метода. Отдельно выведите сводную таблицу средней доли верных пар на сильнейшем уровне (строки — искажения, столбцы — методы) и сводную таблицу средней повторяемости на том же уровне.

In [ ]:
# Ваш код
# Графики: доля верных пар от уровня, по одному графику на искажение, линии — методы
# Сводные таблицы на уровне 3: доля верных пар и повторяемость (строки — искажения, столбцы — методы)

**Вопрос 1.** Какой метод устойчивее всего к каждому виду искажения, а какой теряет верные пары быстрее остальных? Приведите доли верных пар на уровне 3. Всегда ли метод с высокой повторяемостью даёт и много верных пар? Подтвердились ли ваши гипотезы из пункта 5.1?

**Ответ:** _______________

**Вопрос 2.** Как ведёт себя FAST + SIFT при повороте и масштабе по сравнению с SIFT? Что это говорит о вкладе детектора в результат, если дескриптор у обеих комбинаций одинаковый?

**Ответ:** _______________

### **5.6. Проверка на реальных сериях**

Синтетическое искажение — модель реального. Проверим, насколько хороша эта модель. Каждой серии Oxford Affine соответствует синтетическое искажение того же типа:

| Серия | Реальное искажение | Синтетический аналог |
|---|---|---|
| `bikes`, `trees` | расфокусировка объектива | гауссово размытие |
| `ubc` | сжатие JPEG | JPEG |
| `leuven` | уменьшение диафрагмы камеры (затемнение) | затемнение |
| `boat` | приближение камеры и её поворот | поворот и масштаб |

**1. Для каждой из пяти серий словаря `PAIRS`, пар 1 → 2, …, 1 → 6 и каждого метода** вычислите `evaluate_pair` и соберите таблицу `real` со столбцами `серия`, `пара` (2…6), `метод` и метриками. Признаки снимка 1 серии, как и в 5.5, вычисляйте один раз. Выведите, сколько пар из пяти совмещено в каждой серии.

**2. Сравните синтетику и реальность.** Постройте таблицу, где для каждого типа искажения стоят рядом доля верных пар каждого метода на сильнейшем синтетическом уровне (среднее по вашим снимкам) и на паре 1 → 6 соответствующих реальных серий (среднее по сериям).

In [ ]:
# Ваш код
# PAIRS = {'гауссово размытие': ['bikes', 'trees'], 'JPEG': ['ubc'], 'затемнение': ['leuven'], 'поворот и масштаб': ['boat']}
# rows = []
# for seq in sum(PAIRS.values(), []):
#     ...  # признаки снимка 1 — один раз; пары 1 → k для k = 2..6; evaluate_pair
# real = pd.DataFrame(rows)
# Таблица: сколько пар из 5 совмещено (ошибка H < 5)
# Для каждого искажения и метода: доля верных на уровне 3 (synthetic) и на паре 1 → 6 (real)

**Вопрос 3.** Для каких типов искажений синтетика хорошо предсказывает результат на реальных снимках, а для каких — нет? Совпадает ли **порядок** методов (кто лучше, кто хуже)? Предложите объяснение расхождений: чем реальное искажение отличается от его синтетической модели?

**Ответ:** _______________

### **5.7. Скорость и разрешение**

Скорость оценим на полноразмерном кадре с дрона (4000×3000 пикселей) и его копии, уменьшенной в 4 раза по каждой стороне: так видно, как время растёт с размером изображения. Подробнее о замере скорости: [**Теория №7: скорость детекторов на кадре с дрона**](https://colab.research.google.com/drive/1HZSncySLe9Me1pLhDUWMU_W15F8vUZfD#scrollTo=643d849d).

In [ ]:
frame_path = download('https://raw.githubusercontent.com/OpenDroneMap/odm_data_toledo/master/images/DJI_0105.JPG', 'DJI_0105.JPG')
frame_gray = gray(cv2.imread(frame_path))
print(frame_gray.shape)

**1. Для масштабов 0,25 и 1,0** (уменьшение кадра `cv2.resize` с `interpolation=cv2.INTER_AREA`) и каждого метода измерьте функцией `measure_time` время `detect_and_describe` (медиана трёх запусков). Соберите таблицу `speed` со столбцами `мегапиксели`, `метод`, `время, с`, `кадров в секунду`, `точек`. Постройте график времени от числа мегапикселей (логарифмическая шкала времени).

In [ ]:
# Ваш код
# rows = []
# for scale in [0.25, 1.0]:
#     ...  # уменьшить кадр, для каждого метода measure_time(detect_and_describe, img, name)
# speed = pd.DataFrame(rows)
# График: время от числа мегапикселей, логарифмическая шкала времени

**Вопрос 4.** Во сколько раз самый быстрый метод быстрее самого медленного на полном кадре? Какие методы можно использовать для обработки видео 30 кадров в секунду и при каком разрешении? Сопоставьте с качеством из пунктов 5.5–5.6: какой метод даёт лучший компромисс «качество — скорость»?

**Ответ:** _______________

---

## **6. Задание 2. Мультимодальное сопоставление изображений**

Снимки одной сцены в разных спектральных диапазонах выглядят по-разному: в ближнем инфракрасном диапазоне (NIR, 750–1100 нм) растительность ярко-белая, небо и вода тёмные, а многие краски и ткани меняют яркость относительно соседних объектов. Дескрипторы, построенные на перепадах яркости, при этом могут перестать совпадать. Используем набор [**EPFL RGB–NIR Scene Dataset**](https://www.epfl.ch/labs/ivrl/research/downloads/rgb-nir-scene-dataset/): каждая пара снята совмещённо, поэтому эталонная геометрия известна.

### **6.1. Загрузка пар RGB–NIR**

Архив набора занимает 950 МБ, но целиком он не нужен. Библиотека `remotezip` читает из архива по сети только нужные файлы. Функция `load_nir_pair` возвращает снимок RGB (в формате BGR) и снимок NIR (оттенки серого) и сохраняет их в `data/nir`, чтобы не скачивать повторно.

In [ ]:
nir_zip = RemoteZip('https://ivrlwww.epfl.ch/supplementary_material/cvpr11/nirscene1.zip')
NIR_NAMES = nir_zip.namelist()
NIR_CATEGORIES = sorted({name.split('/')[1] for name in NIR_NAMES if name.count('/') == 2})
os.makedirs('data/nir', exist_ok=True)

def nir_ids(category):
    """Номера пар категории."""
    return sorted({re.findall(r'(\d+)_rgb', n)[0] for n in NIR_NAMES if f'/{category}/' in n and n.endswith('_rgb.tiff')})

def load_nir_pair(category, pair_id):
    """Снимок RGB (BGR) и снимок NIR (оттенки серого) пары."""
    images = []
    for band, flag in [('rgb', cv2.IMREAD_COLOR), ('nir', cv2.IMREAD_GRAYSCALE)]:
        path = f'data/nir/{category}_{pair_id}_{band}.tiff'
        if not os.path.exists(path):
            with open(path, 'wb') as file:
                file.write(nir_zip.read(f'nirscene1/{category}/{pair_id}_{band}.tiff'))
        images.append(cv2.imread(path, flag))
    return images

print({category: len(nir_ids(category)) for category in NIR_CATEGORIES})

Пары RGB и NIR совмещены, поэтому сопоставлять их «как есть» слишком просто: точка ищется на том же месте. Функция `make_test_pair` переносит снимок NIR известной гомографией (поворот на 12°, масштаб 0,85 и небольшая перспектива — одинаковые для всех пар) и возвращает эталонную матрицу. Теперь задача сопоставления настоящая, а правильность каждой пары по-прежнему проверяется точно.

In [ ]:
def make_test_pair(img1, img2):
    """img1 без изменений, img2 перенесённый известной гомографией, и эта гомография."""
    h, w = img1.shape[:2]
    corners = np.float32([[0, 0], [w, 0], [w, h], [0, h]])
    A = np.vstack([cv2.getRotationMatrix2D((w / 2, h / 2), 12, 0.85), [0, 0, 1]])
    moved = cv2.perspectiveTransform(corners.reshape(-1, 1, 2), A).reshape(-1, 2) + np.float32([[10, -8], [-12, 6], [8, 10], [-6, -10]])
    H = cv2.getPerspectiveTransform(corners, moved)
    return img1, cv2.warpPerspective(img2, H, (w, h)), H

**1. Выберите 2–3 категории** из списка `NIR_CATEGORIES` (например, контрастную городскую, интерьерную и природную) и запишите их в список `CATEGORIES`. Покажите для каждой категории одну пару: снимок RGB и снимок NIR.

In [ ]:
# Ваш код
# CATEGORIES = [...]          # 2–3 категории из NIR_CATEGORIES
# PAIRS_PER_CATEGORY = 10     # число пар каждой категории для эксперимента
# Покажите по одной паре RGB / NIR для каждой категории

**Вопрос 1.** Какие объекты выглядят в NIR иначе, чем в видимом диапазоне (светлее, темнее, теряют контраст с соседями)? Какие различия, по-вашему, сильнее всего помешают дескрипторам?

**Ответ:** _______________

### **6.2. Предобработка**

**CLAHE** (Contrast Limited Adaptive Histogram Equalization) выравнивает гистограмму в небольших блоках изображения и ограничивает усиление контраста, чтобы не усиливать шум. Растягивая местные перепады яркости, оно делает их в разных диапазонах более похожими.

**1. Напишите функцию `preprocess(img, mode)`**, которая для полутонового изображения возвращает: при `mode='нет'` — изображение без изменений; при `mode='CLAHE'` — результат `cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(img)`. Покажите снимок NIR одной пары в двух вариантах.

In [ ]:
# Ваш код
# PREPROCESSING = ['нет', 'CLAHE']
# def preprocess(img, mode): ...
# Покажите снимок NIR в двух вариантах предобработки

### **6.3. Сравнение методов и предобработки**

**1. Для каждой выбранной категории, первых `PAIRS_PER_CATEGORY` пар, каждого варианта предобработки и каждого метода**: переведите RGB в оттенки серого, примените `preprocess` к обоим снимкам, получите тестовую пару `make_test_pair(rgb_gray, nir)` и вычислите `evaluate_pair`. Соберите таблицу `multimodal` со столбцами `категория`, `пара`, `предобработка`, `метод` и метриками. Выведите две сводные таблицы: долю совмещённых пар (`ошибка H < 5`) и среднюю долю верных пар (строки — метод, столбцы — предобработка). Эксперимент занимает около полутора минут. Десять пар на категорию — разумный минимум: при пяти парах одна пара меняет долю совмещённых на 0,2, и различия между вариантами предобработки тонут в случайном разбросе.

In [ ]:
# Ваш код
# rows = []
# for category in CATEGORIES:
#     for pair_id in nir_ids(category)[:PAIRS_PER_CATEGORY]:
#         ...  # загрузить пару; для каждой предобработки — make_test_pair; для каждого метода — evaluate_pair
# multimodal = pd.DataFrame(rows)
# Сводные таблицы: доля совмещённых пар и средняя доля верных пар (метод × предобработка); по категориям для CLAHE

**2. Покажите пары точек** (функция `draw_checked`) для одной трудной пары: без предобработки и с CLAHE, методом, который сильнее всего выигрывает от предобработки.

In [ ]:
# Ваш код
# Выберите трудную пару и метод; покажите draw_checked без предобработки и с CLAHE

**Вопрос 2.** Какой метод лучше всего сопоставляет RGB и NIR? Как предобработка меняет долю совмещённых пар и долю верных пар у разных методов? Для какой категории сопоставление труднее всего и почему?

**Ответ:** _______________

---

## **7. Задание 3. Методы в специальных доменах**

**Выберите и выполните один из трёх вариантов.** Варианты независимы: данные для каждого загружаются в его разделе.

| Вариант | Задача | Данные | Главный вопрос |
|---|---|---|---|
| **7.1. Медицина** | Регистрация (совмещение) снимков глазного дна | FIRE, 134 пары с эталонными точками | Какой метод надёжнее совмещает снимки при малом перекрытии и при изменениях сетчатки со временем? |
| **7.2. Повторяющиеся структуры** | Сопоставление на кирпичной кладке и листве | Oxford Affine: `wall`, `trees` против `graf` | Действительно ли повторяющийся узор сбивает дескрипторы и как на нём работает тест Лоу? |
| **7.3. Аэросъёмка** | Привязка кадров с дрона к спутниковой подложке | OpenDroneMap Toledo, Esri World Imagery | На каких кадрах и каким методом привязка работает, а где ломается? |

### **7.1. Вариант 1. Регистрация медицинских снимков (FIRE)**

Регистрация — приведение двух снимков к общей системе координат. Для снимков глазного дна она нужна, чтобы сравнивать обследования одного пациента в разное время или собирать мозаику сетчатки из нескольких снимков. Набор [**FIRE**](https://projects.ics.forth.gr/cvrl/fire/) содержит 134 пары снимков 2912×2912 пикселей трёх групп: **S** — снимки с большим перекрытием; **P** — с малым перекрытием (пересекается лишь часть поля зрения); **A** — снимки, сделанные с интервалом во времени, с анатомическими изменениями. Для каждой пары вручную размечено 10 соответствующих точек — это эталон.

Сетчатка — изогнутая поверхность, но при небольшом угле зрения её совмещение хорошо описывается гомографией. Ошибку регистрации измеряют на эталонных точках: переносят точки первого снимка найденной гомографией и усредняют расстояние до их пар на втором снимке. Регистрацию считают успешной, если средняя ошибка меньше 25 пикселей (около 1% стороны снимка).

Готовый код загружает архив (277 МБ) и распаковывает его. Функция `load_fire_pair` возвращает два снимка, уменьшенные в 4 раза (728×728 пикселей), — зелёный канал после CLAHE: в нём сосуды сетчатки контрастнее всего — и массив эталонных точек в пикселях полного разрешения: столбцы `x1, y1, x2, y2`. Маска `fire_mask` закрывает чёрную рамку вокруг поля зрения, чтобы детекторы не находили точки на её крае. Функция `checkerboard` собирает шахматную мозаику из двух снимков: по стыкам клеток видно, совпадают ли сосуды.

In [ ]:
fire_archive = download('https://projects.ics.forth.gr/cvrl/fire/FIRE.7z', 'FIRE.7z')
if not os.path.isdir('data/FIRE'):
    with py7zr.SevenZipFile(fire_archive) as archive:
        archive.extractall('data')

FIRE_SCALE = 0.25
FIRE_IDS = sorted(os.path.basename(p).split('_')[2] for p in glob.glob('data/FIRE/Ground Truth/control_points_*_1_2.txt'))
fire_mask = cv2.resize(cv2.imread('data/FIRE/Masks/mask.png', cv2.IMREAD_GRAYSCALE), None, fx=FIRE_SCALE, fy=FIRE_SCALE)
fire_clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

def load_fire_pair(pair_id):
    """Два уменьшенных снимка (зелёный канал + CLAHE) и эталонные точки x1, y1, x2, y2 в полном разрешении."""
    images = [fire_clahe.apply(cv2.resize(cv2.imread(f'data/FIRE/Images/{pair_id}_{i}.jpg'), None, fx=FIRE_SCALE, fy=FIRE_SCALE,
                                          interpolation=cv2.INTER_AREA)[:, :, 1]) for i in (1, 2)]
    points = np.loadtxt(f'data/FIRE/Ground Truth/control_points_{pair_id}_1_2.txt')
    return images[0], images[1], points

def checkerboard(a, b, cell=60):
    """Шахматная мозаика: клетки поочерёдно из снимков a и b."""
    checker = (np.indices(a.shape) // cell).sum(axis=0) % 2 == 0
    return np.where(checker, a, b)

print(len(FIRE_IDS), 'пар:', {g: sum(p.startswith(g) for p in FIRE_IDS) for g in 'SPA'})

**1. Напишите функцию `register(a, b, name)`**: точки и дескрипторы на обоих снимках с маской `fire_mask` (`detect_and_describe(..., mask=fire_mask)`), сопоставление с тестом Лоу и гомография методом RANSAC (порог 3 пикселя). Функция возвращает гомографию в координатах уменьшенных снимков или `None`, если пар меньше четырёх или RANSAC не нашёл модель.

**2. Напишите функцию `fire_error(H, points)`**, которая переводит эталонные точки первого снимка в масштаб уменьшенных снимков (умножение на `FIRE_SCALE`), переносит их гомографией `H`, возвращает в полное разрешение и вычисляет среднее расстояние до точек второго снимка. Для `H = None` верните `np.inf`.

Подробнее о гомографии и RANSAC: [**Теория №7: 2.8. Гомография и RANSAC**](https://colab.research.google.com/drive/1HZSncySLe9Me1pLhDUWMU_W15F8vUZfD#scrollTo=e584fd36).

In [ ]:
# Ваш код
# def register(a, b, name): ...       # -> гомография 3×3 или None
# def fire_error(H, points): ...      # -> средняя ошибка на эталонных точках в пикселях полного разрешения
# Проверьте на паре 'S01' методом SIFT

**3. Выполните регистрацию всех 134 пар каждым методом** и соберите таблицу `fire` со столбцами `пара`, `группа` (первая буква номера пары), `метод`, `ошибка`. Выведите долю успешных регистраций (ошибка < 25 пикселей) и медианную ошибку по группам и методам. Эксперимент занимает 3–4 минуты.

In [ ]:
# Ваш код
# rows = []
# for pair_id in FIRE_IDS:
#     ...  # загрузить пару; для каждого метода — register и fire_error
# fire = pd.DataFrame(rows)
# Таблицы по группам S, P, A: доля успешных регистраций и медианная ошибка

**4. Покажите результат регистрации** одной успешно совмещённой пары группы P лучшим методом: мозаику `checkerboard` из снимка 1 и снимка 2, перенесённого обратной гомографией (`cv2.warpPerspective(b, np.linalg.inv(H), ...)`), рядом с мозаикой без регистрации.

In [ ]:
# Ваш код
# Выберите пару группы P; H = register(a, b, лучший метод)
# Мозаика checkerboard без регистрации и после регистрации

**Вопрос 1 (вариант 1).** Как меняется доля успешных регистраций от группы S к группам P и A у каждого метода? Какой метод вы рекомендуете для регистрации снимков сетчатки и почему? Свяжите результат с устройством методов: какие структуры на снимке глазного дна служат ключевыми точками и что происходит с ними при малом перекрытии?

**Ответ:** _______________

### **7.2. Вариант 2. Повторяющиеся структуры: проверка интуиции**

Интуиция подсказывает, что на кирпичной стене, плитке или листве многие окрестности похожи друг на друга, поэтому ближайшим дескриптором часто окажется не та же точка, а соседний элемент узора. Проверим эту гипотезу на сериях Oxford Affine `wall` (кирпичная кладка) и `trees` (листва) в сравнении с `graf` (граффити с крупным неповторяющимся рисунком). Серии `wall` и `graf` сняты с одинаковой сменой ракурса, поэтому различия между ними связаны прежде всего с видом поверхности. Эталонные гомографии известны во всех трёх сериях. Подробнее о тесте Лоу и проверке пар: [**Теория №7: тест Лоу**](https://colab.research.google.com/drive/1HZSncySLe9Me1pLhDUWMU_W15F8vUZfD#scrollTo=7795d677), [**Теория №7: проверка пар по эталонной гомографии**](https://colab.research.google.com/drive/1HZSncySLe9Me1pLhDUWMU_W15F8vUZfD#scrollTo=69332f28).

**1. Для пары 1 → 3 каждой из серий `wall`, `trees`, `graf` и методов SIFT и ORB** найдите для каждой точки снимка 1 два ближайших дескриптора. Для этого напишите функцию `nearest_pairs(feat1, feat2, H, norm)`: два ближайших дескриптора (`knnMatch`, `k=2`) для каждой точки снимка 1 и два массива — `correct` (верна ли ближайшая пара: ошибка переноса меньше 3 пикселей) и `ratio` (отношение `d1 / d2`). Для порогов теста Лоу от 0,5 до 0,95 с шагом 0,05 вычислите число верных принятых пар и их долю (Precision). Постройте оба графика и выведите таблицу при порогах 0,7 и 0,8.

In [ ]:
# Ваш код
# def nearest_pairs(feat1, feat2, H, norm): ...   # -> correct (bool), ratio (d1/d2) для каждой точки снимка 1
# Для серий wall, trees, graf (пара 1 → 3) и методов SIFT, ORB:
#   для порогов 0,5 … 0,95 — число верных принятых пар и Precision
# Графики и таблица при порогах 0,7 и 0,8

**Вопрос 1 (вариант 2).** Подтвердилась ли гипотеза о том, что повторяющийся узор сбивает дескрипторы? Сравните кривые Precision и число верных пар для `wall`, `trees` и `graf`. Объясните результат, опираясь на размер окрестности, которую описывают SIFT и ORB, и на то, насколько одинаковы кирпичи на самом деле.

**Ответ:** _______________

### **7.3. Вариант 3. Привязка кадров с дрона к спутниковой подложке**

В теории ([**Теория №7: 3.2. Привязка кадра с дрона**](https://colab.research.google.com/drive/1HZSncySLe9Me1pLhDUWMU_W15F8vUZfD#scrollTo=36d3593f)) один кадр с дрона был привязан к подложке Esri World Imagery: кадр приводится к разрешению подложки, сопоставляется с ней, и найденное подобие даёт географические координаты каждого пикселя. Проверим этот подход на нескольких кадрах того же полёта ([OpenDroneMap Toledo](https://github.com/OpenDroneMap/odm_data_toledo)): над зданиями и парковкой, над дорожками и над полями. Подложка снята в другой сезон, поэтому трава и деревья на ней выглядят иначе, чем на кадрах.

Эталона в виде гомографии здесь нет, но есть два независимых способа проверить привязку: **расстояние от центра кадра до координат камеры по GPS** (камера смотрит вертикально вниз) и **ширина кадра на местности**, вычисленная по найденному масштабу, в сравнении с шириной по высоте съёмки и параметрам камеры.

Готовые функции взяты из теории: чтение координат, высоты и фокусного расстояния из EXIF, перевод между географическими и пиксельными координатами Web Mercator, загрузка мозаики тайлов вокруг точки, расстояние между близкими точками и `prepare_frame` — шаги 1–2 раздела 3.2 теории: кадр уменьшается так, чтобы его пиксель совпал с пикселем подложки (ширина кадра на местности = высота × ширина матрицы / фокусное расстояние). Список `DRONE_FRAMES` — кадры, из которых можно выбирать.

In [ ]:
TOLEDO_URL = 'https://raw.githubusercontent.com/OpenDroneMap/odm_data_toledo/master/images/'
DRONE_FRAMES = ['1JI_0062', '1JI_0068', '1JI_0074', '1JI_0080', '1JI_0086', '1JI_0092',
                'DJI_0063', 'DJI_0075', 'DJI_0081', 'DJI_0093', 'DJI_0099', 'DJI_0105']
SENSOR_WIDTH_MM = 6.17   # ширина матрицы 1/2,3″ камер DJI этого полёта
ZOOM, RADIUS = 19, 3

def read_exif(path):
    """Широта, долгота, высота съёмки (м) и фокусное расстояние (мм) из EXIF."""
    exif = Image.open(path).getexif()
    gps = exif.get_ifd(0x8825)
    to_degrees = lambda v, ref: (float(v[0]) + float(v[1]) / 60 + float(v[2]) / 3600) * (-1 if ref in ('S', 'W') else 1)
    return to_degrees(gps[2], gps[1]), to_degrees(gps[4], gps[3]), float(gps[6]), float(exif.get_ifd(0x8769)[0x920A])

def lonlat_to_pixel(lon, lat, zoom=ZOOM):
    n = 256 * 2 ** zoom
    return (lon + 180) / 360 * n, (1 - math.asinh(math.tan(math.radians(lat))) / math.pi) / 2 * n

def pixel_to_lonlat(x, y, zoom=ZOOM):
    n = 256 * 2 ** zoom
    return x / n * 360 - 180, math.degrees(math.atan(math.sinh(math.pi * (1 - 2 * y / n))))

def load_basemap(lon, lat, zoom=ZOOM, radius=RADIUS):
    """Мозаика (2·radius + 1)² тайлов Esri World Imagery вокруг точки, пиксельные координаты её угла и размер пикселя, м."""
    cx, cy = lonlat_to_pixel(lon, lat, zoom)
    tx, ty = int(cx // 256), int(cy // 256)
    rows = []
    for j in range(ty - radius, ty + radius + 1):
        row = []
        for i in range(tx - radius, tx + radius + 1):
            url = f'https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{zoom}/{j}/{i}'
            row.append(cv2.imread(download(url, f'esri_World_Imagery_{zoom}_{j}_{i}.jpg')))
        rows.append(np.hstack(row))
    m_per_px = 156543.03392 * math.cos(math.radians(lat)) / 2 ** zoom
    return np.vstack(rows), np.array([(tx - radius) * 256, (ty - radius) * 256]), m_per_px

def ground_distance(lon1, lat1, lon2, lat2):
    """Расстояние в метрах между близкими точками."""
    dx = math.radians(lon2 - lon1) * 6371000 * math.cos(math.radians(lat1))
    dy = math.radians(lat2 - lat1) * 6371000
    return math.hypot(dx, dy)

def prepare_frame(path):
    """Кадр в оттенках серого, уменьшенный до разрешения подложки, подложка вокруг точки GPS и параметры привязки."""
    lat, lon, altitude, focal_mm = read_exif(path)
    basemap, origin, m_per_px = load_basemap(lon, lat)
    frame = cv2.imread(path)
    width_m = altitude * SENSOR_WIDTH_MM / focal_mm   # ширина кадра на местности
    scale = width_m / m_per_px / frame.shape[1]
    small = gray(cv2.resize(frame, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA))
    return small, gray(basemap), {'lat': lat, 'lon': lon, 'width_m': width_m, 'origin': origin, 'm_per_px': m_per_px}

**1. Выберите не меньше пяти кадров** из `DRONE_FRAMES`, загрузите их (`download(TOLEDO_URL + имя + '.JPG', имя + '.JPG')`) и покажите уменьшенные копии в одной сетке. Постарайтесь, чтобы среди них были кадры с постройками и кадры, где видны в основном поля и деревья.

**2. Напишите функцию `georeference(path, name)`** по шагам 3–4 раздела 3.2 теории ([**Теория №7: сопоставление кадра с подложкой**](https://colab.research.google.com/drive/1HZSncySLe9Me1pLhDUWMU_W15F8vUZfD#scrollTo=505fc0a3)): подготовить кадр и подложку функцией `prepare_frame`; найти точки и сопоставить кадр с подложкой методом `name`; оценить подобие `cv2.estimateAffinePartial2D` с RANSAC (порог 5 пикселей); перенести центр кадра подобием на подложку и перевести в географические координаты (`pixel_to_lonlat` от пиксельных координат подложки плюс `origin`). Функция возвращает словарь: `инлайеров`, `центр — GPS, м`, `ширина, м` (по найденному масштабу), `ожидаемая ширина, м`. Если подобие не найдено, верните `np.inf` для расстояния и ширины.

In [ ]:
# Ваш код
# FRAMES = [...]   # не меньше пяти кадров из DRONE_FRAMES
# paths = {frame: download(TOLEDO_URL + frame + '.JPG', frame + '.JPG') for frame in FRAMES}
# Покажите кадры в сетке

# def georeference(path, name): ...   # -> словарь: инлайеров, центр — GPS, ширина, ожидаемая ширина
# Проверьте на одном кадре

**3. Привяжите все выбранные кадры всеми методами.** Соберите таблицу `drone` и добавьте столбец `успех`: центр кадра ближе 15 м к точке GPS и ширина отличается от ожидаемой меньше чем на 10%. Выведите таблицы успехов и расстояний «центр — GPS» (строки — кадры, столбцы — методы).

In [ ]:
# Ваш код
# rows = []
# for frame in FRAMES:
#     for name in METHOD_NAMES:
#         ...  # georeference
# drone = pd.DataFrame(rows)
# drone['успех'] = ...   # центр ближе 15 м к GPS и ширина отличается от ожидаемой меньше чем на 10 %
# Таблицы: успех, расстояние центр — GPS, число инлайеров

**Вопрос 1 (вариант 3).** На каких кадрах привязка удалась большинством методов, а на каких не удалась ни одним? Что изображено на этих кадрах? Какой метод вы рекомендуете для этой задачи и почему? Почему здесь используется модель подобия, а не гомография (теория, раздел 2.8, таблица моделей)?

**Ответ:** _______________

---

## **8. Итоги и рекомендации**

**1. Заполните таблицу рекомендаций** по своим результатам: для каждого условия укажите лучший метод (или два равноценных) и число из своих таблиц, на котором основан выбор. Сравните с таблицей «Какой метод выбрать» из теории ([**Теория №7: 2.9. Сравнительный анализ**](https://colab.research.google.com/drive/1HZSncySLe9Me1pLhDUWMU_W15F8vUZfD#scrollTo=49251e77)): где ваши выводы совпали, а где уточнили или опровергли её?

| Условия | Рекомендуемый метод | Обоснование (число из ваших результатов) |
|---|---|---|
| Размытие | ... | ... |
| Сжатие JPEG и затемнение | ... | ... |
| Поворот и масштаб | ... | ... |
| Обработка видео в реальном времени | ... | ... |
| RGB ↔ NIR | ... | ... |
| Ваш вариант Задания 3 | ... | ... |

**Ответ (таблица рекомендаций):** _______________

**2. Общий вывод.** Есть ли метод, который лучше остальных во всех условиях? Какой главный урок о проверке методов вы вынесли из сравнения синтетических и реальных искажений? Ответьте 5–7 предложениями.

**Вывод:** _______________

---

## **9. Что сдавать**

1. Выполненный блокнот со всеми выводами ячеек: таблицами и графиками всех трёх заданий (в Задании 3 — выбранного варианта).
2. Заполненные гипотезы (5.1), ответы на вопросы всех заданий и таблицу рекомендаций (раздел 8).
3. Ответы должны опираться на числа из ваших таблиц: «AKAZE лучше» без числа не засчитывается.

---

## **10. Дополнительные задания (выполняются по желанию)**

1. **Кривые Precision–Recall и ROC по эталону.** Для пар RGB ↔ NIR с CLAHE найдите для каждой точки снимка 1 два ближайших дескриптора и запишите метку (верна ли ближайшая пара по эталонной гомографии) и оценку уверенности `1 − d1/d2`. Постройте кривые `precision_recall_curve` и `roc_curve` из `sklearn.metrics` для каждого метода и сравните ROC-AUC. Совпадает ли порядок методов с долей совмещённых пар из 6.3?
2. **Снимок и карта.** Загрузите тайлы Esri World Imagery и Esri World Street Map одного района (адрес тайла: `https://server.arcgisonline.com/ArcGIS/rest/services/<слой>/MapServer/tile/<z>/<y>/<x>`, [**Теория №7: загрузка подложки**](https://colab.research.google.com/drive/1HZSncySLe9Me1pLhDUWMU_W15F8vUZfD#scrollTo=83fef17e)) и сопоставьте их функцией `evaluate_pair` после `make_test_pair`. Сколько верных пар находят методы и почему?
3. **USAC_MAGSAC вместо RANSAC.** Повторите 6.3 для варианта CLAHE, заменив в `evaluate_pair` метод `cv2.RANSAC` на `cv2.USAC_MAGSAC`. Изменилась ли доля совмещённых пар RGB ↔ NIR?
4. **FLANN на большом числе точек.** Для полного кадра с дрона и 20 000 точек SIFT сравните время сопоставления `cv2.BFMatcher` и `cv2.FlannBasedMatcher` ([**Теория №7: 2.7. Сопоставление**](https://colab.research.google.com/drive/1HZSncySLe9Me1pLhDUWMU_W15F8vUZfD#scrollTo=fe29c5b2)). С какого числа точек FLANN становится быстрее?
5. **Второй вариант Задания 3.** Выполните ещё один вариант и сравните, совпадают ли рекомендации для разных доменов.